# University Query Management System
## RAG Based Query System for Engineering College

This project builds a query management system for an engineering college where students and staff can ask questions about hostel, mess, fees, placements, anti-ragging policies and more, and get accurate answers with citations.

The system is built using RAG (Retrieval Augmented Generation) which means it first finds relevant information from our university database and then uses an LLM to generate a clear answer based on that information.

**Tools used:**
- Sentence Transformers (HuggingFace) for converting text into vectors
- FAISS as the vector database for fast similarity search
- Groq LLM for generating natural language answers
- Streamlit for the web interface

### Phase 1 — Installing Required Libraries

In [ ]:
!pip install -q chromadb google-generativeai
print("Done.")

In [ ]:
!pip install -q faker
print("Done.")

In [ ]:
!pip install -q google-genai
print("Done.")

In [ ]:
!pip install -q faiss-cpu scikit-learn
print("Done.")

In [ ]:
!pip install -q groq
print("Done.")

In [ ]:
from groq import Groq
client = Groq(api_key="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")

for model in client.models.list().data:
    print(model.id)

In [ ]:
!pip install -q sentence-transformers
print("Done.")

### Phase 2 — Data Generation

Before building the RAG system, we need data. I created synthetic but realistic data for 16 different modules of an engineering college. This includes 1000 student records along with information about hostel, mess menu, anti-ragging policy, faculty, fees, admissions, scholarships, library, clubs, transport, placements, grievance cell, health center, rules and labs.


In [ ]:
# ============================================================
# PHASE 2 — DATA GENERATION
# Creating and preparing the university knowledge base
# ============================================================

import pandas as pd
import random
from faker import Faker

fake = Faker('en_IN')
random.seed(42)

branches = ['CSE', 'ECE', 'ME', 'CE', 'EE', 'IT', 'CH']
years = [1, 2, 3, 4]
hostels = ['Hostel A', 'Hostel B', 'Hostel C', 'Hostel D', 'Day Scholar']

# 1. STUDENT RECORDS
students = []
for i in range(1, 1001):
    branch = random.choice(branches)
    year = random.choice(years)
    roll = f"{year}{branch}{str(i).zfill(4)}"
    students.append({
        'student_id': i, 'roll_number': roll, 'name': fake.name(),
        'branch': branch, 'year': year,
        'gender': random.choice(['Male', 'Female']),
        'blood_group': random.choice(['A+','A-','B+','B-','O+','O-','AB+','AB-']),
        'phone': fake.phone_number(), 'email': fake.email(),
        'address': fake.address().replace('\n', ', '),
        'hostel': random.choice(hostels),
        'cgpa': round(random.uniform(5.0, 10.0), 2),
        'attendance': random.randint(60, 100),
        'scholarship': random.choice(['Yes', 'No']),
        'fee_paid': random.choice(['Yes', 'No']),
    })
pd.DataFrame(students).to_csv('students.csv', index=False)
print("Student records created.")

# 2. HOSTEL
hostel_data = []
for h in ['Hostel A', 'Hostel B', 'Hostel C', 'Hostel D']:
    hostel_data.append({
        'hostel_name': h, 'warden': fake.name(),
        'warden_contact': fake.phone_number(),
        'total_rooms': random.randint(50,100),
        'capacity_per_room': 2,
        'mess_attached': random.choice(['Yes','No']),
        'wifi': 'Yes', 'laundry': 'Yes',
        'curfew_time': '10:00 PM',
        'rules': 'No smoking, no alcohol, visitors allowed till 8 PM'
    })
pd.DataFrame(hostel_data).to_csv('hostel.csv', index=False)
print("Hostel data created.")

# 3. MESS MENU
mess = [
    {'day': 'Monday', 'breakfast': 'Poha, Tea', 'lunch': 'Dal, Rice, Roti, Sabzi', 'dinner': 'Paneer, Rice, Roti, Salad'},
    {'day': 'Tuesday', 'breakfast': 'Idli, Sambar', 'lunch': 'Rajma, Rice, Roti', 'dinner': 'Chole, Roti, Rice'},
    {'day': 'Wednesday', 'breakfast': 'Bread, Butter, Omelette', 'lunch': 'Dal Fry, Rice, Roti', 'dinner': 'Mixed Veg, Roti, Rice'},
    {'day': 'Thursday', 'breakfast': 'Upma, Tea', 'lunch': 'Kadhi, Rice, Roti', 'dinner': 'Egg Curry, Roti, Rice'},
    {'day': 'Friday', 'breakfast': 'Puri, Sabzi', 'lunch': 'Dal Makhani, Rice, Roti', 'dinner': 'Paneer Butter Masala, Naan'},
    {'day': 'Saturday', 'breakfast': 'Dosa, Chutney', 'lunch': 'Biryani, Raita', 'dinner': 'Dal, Rice, Roti, Kheer'},
    {'day': 'Sunday', 'breakfast': 'Chole Bhature', 'lunch': 'Special Thali', 'dinner': 'Fried Rice, Manchurian'},
]
pd.DataFrame(mess).to_csv('mess.csv', index=False)
print("Mess menu created.")

# 4. ANTI-RAGGING
anti_ragging = [{'policy': 'Zero tolerance for ragging',
    'helpline': '1800-180-5522',
    'email': 'antiragging@university.ac.in',
    'committee_head': 'Dr. R.K. Sharma',
    'contact': '9876500001',
    'punishment': 'Expulsion and FIR',
    'definition': 'Any act that causes physical or mental harm to a student',
    'reporting': 'Report to warden, HOD, or anti-ragging committee immediately'}]
pd.DataFrame(anti_ragging).to_csv('anti_ragging.csv', index=False)
print("Anti-ragging data created.")

# 5. ACADEMIC INFO
academic = []
for branch in branches:
    academic.append({
        'branch': branch, 'total_semesters': 8,
        'credits_required': 160, 'min_attendance': '75%',
        'grading_system': '10 point CGPA',
        'backlog_policy': 'Max 2 backlogs allowed per semester',
        'lateral_entry': 'Available for diploma holders in 2nd year'
    })
pd.DataFrame(academic).to_csv('academic_info.csv', index=False)
print("Academic info created.")

# 6. EXAMS
exams = [
    {'exam_type': 'Mid Semester', 'month': 'September', 'marks': 30, 'duration': '1.5 hours'},
    {'exam_type': 'End Semester', 'month': 'November', 'marks': 70, 'duration': '3 hours'},
    {'exam_type': 'Mid Semester', 'month': 'February', 'marks': 30, 'duration': '1.5 hours'},
    {'exam_type': 'End Semester', 'month': 'April', 'marks': 70, 'duration': '3 hours'},
]
pd.DataFrame(exams).to_csv('exams.csv', index=False)
print("Exam data created.")

# 7. FACULTY WITH HOD SUMMARY
faculty = []
for branch in branches:
    hod_name = fake.name()
    faculty.append({
        'name': hod_name,
        'designation': 'HOD',
        'branch': branch,
        'summary': f'The HOD of {branch} department is {hod_name}',
        'email': fake.email(),
        'contact': fake.phone_number(),
        'cabin': f'Block A, Room {random.randint(100,200)}',
        'specialization': 'Various'
    })
    for _ in range(4):
        faculty.append({
            'name': fake.name(),
            'designation': random.choice(['Professor','Associate Professor','Assistant Professor']),
            'branch': branch,
            'summary': '',
            'email': fake.email(),
            'contact': fake.phone_number(),
            'cabin': f'Block {random.choice(["A","B","C"])}, Room {random.randint(100,300)}',
            'specialization': 'Various'
        })
pd.DataFrame(faculty).to_csv('faculty.csv', index=False)
print("Faculty directory created.")

# 8. FEES
fees = []
for branch in branches:
    fees.append({
        'branch': branch, 'annual_tuition_fee': 85000,
        'hostel_fee': 45000, 'mess_fee': 30000,
        'exam_fee': 2000, 'library_fee': 1000,
        'total_annual': 163000,
        'due_date_sem1': 'July 31', 'due_date_sem2': 'January 31',
        'payment_mode': 'Online/DD/Cash',
        'late_fine': '100 per day after due date'
    })
pd.DataFrame(fees).to_csv('fees.csv', index=False)
print("Fee structure created.")

# 9. ADMISSIONS
admissions = [
    {'program': 'B.Tech', 'duration': '4 years',
     'eligibility': '10+2 with PCM, min 60%',
     'entrance_exam': 'JEE Main / State CET',
     'seats_per_branch': 60,
     'admission_process': 'Merit based counselling',
     'documents_required': '10th, 12th marksheet, Transfer certificate, Character certificate'},
    {'program': 'M.Tech', 'duration': '2 years',
     'eligibility': 'B.Tech with min 60%',
     'entrance_exam': 'GATE', 'seats_per_branch': 18,
     'admission_process': 'GATE score based',
     'documents_required': 'B.Tech marksheets, GATE scorecard'},
]
pd.DataFrame(admissions).to_csv('admissions.csv', index=False)
print("Admissions data created.")

# 10. SCHOLARSHIPS
scholarships = [
    {'name': 'Merit Scholarship', 'eligibility': 'CGPA above 9.0', 'amount': 20000, 'type': 'Annual'},
    {'name': 'SC/ST Scholarship', 'eligibility': 'SC/ST category students', 'amount': 50000, 'type': 'Annual'},
    {'name': 'OBC Scholarship', 'eligibility': 'OBC category, family income below 3 lakh', 'amount': 25000, 'type': 'Annual'},
    {'name': 'Sports Scholarship', 'eligibility': 'National level sports achievement', 'amount': 15000, 'type': 'Annual'},
    {'name': 'EWS Scholarship', 'eligibility': 'Economically weaker section', 'amount': 30000, 'type': 'Annual'},
]
pd.DataFrame(scholarships).to_csv('scholarships.csv', index=False)
print("Scholarships data created.")

# 11. LIBRARY
library = [{'facility': 'Central Library',
    'timing': '8 AM to 10 PM',
    'books_available': 25000,
    'digital_resources': 'Yes',
    'borrowing_limit': '3 books for 14 days',
    'fine': '2 rupees per day per book',
    'librarian': fake.name(),
    'contact': fake.phone_number(),
    'location': 'Block C, Ground Floor'}]
pd.DataFrame(library).to_csv('library.csv', index=False)
print("Library data created.")

# 12. CLUBS AND SPORTS
clubs = [
    {'name': 'Coding Club', 'type': 'Technical', 'faculty_advisor': fake.name(), 'meeting': 'Every Saturday 3 PM'},
    {'name': 'Robotics Club', 'type': 'Technical', 'faculty_advisor': fake.name(), 'meeting': 'Every Friday 4 PM'},
    {'name': 'NSS', 'type': 'Social', 'faculty_advisor': fake.name(), 'meeting': 'Every Sunday 9 AM'},
    {'name': 'Cricket Team', 'type': 'Sports', 'faculty_advisor': fake.name(), 'meeting': 'Daily 6 AM'},
    {'name': 'Football Team', 'type': 'Sports', 'faculty_advisor': fake.name(), 'meeting': 'Daily 6 AM'},
    {'name': 'Cultural Club', 'type': 'Cultural', 'faculty_advisor': fake.name(), 'meeting': 'Every Wednesday 5 PM'},
    {'name': 'Debate Club', 'type': 'Literary', 'faculty_advisor': fake.name(), 'meeting': 'Every Tuesday 4 PM'},
]
pd.DataFrame(clubs).to_csv('clubs.csv', index=False)
print("Clubs and sports data created.")

# 13. TRANSPORT
transport = [
    {'route': 'Route 1', 'areas': 'City Center, Bus Stand', 'departure': '7:30 AM', 'return': '5:00 PM', 'fee': 8000},
    {'route': 'Route 2', 'areas': 'Railway Station, Civil Lines', 'departure': '7:45 AM', 'return': '5:00 PM', 'fee': 9000},
    {'route': 'Route 3', 'areas': 'Cantonment, Sadar', 'departure': '7:15 AM', 'return': '5:15 PM', 'fee': 7500},
    {'route': 'Route 4', 'areas': 'Aashiyana, Indira Nagar', 'departure': '7:30 AM', 'return': '5:00 PM', 'fee': 8500},
]
pd.DataFrame(transport).to_csv('transport.csv', index=False)
print("Transport data created.")

# 14. PLACEMENTS
placements = [
    {'company': 'TCS', 'package_lpa': 3.5, 'role': 'Software Engineer', 'branches_eligible': 'CSE, IT, ECE', 'year': 2024},
    {'company': 'Infosys', 'package_lpa': 3.6, 'role': 'Systems Engineer', 'branches_eligible': 'CSE, IT, ECE', 'year': 2024},
    {'company': 'Wipro', 'package_lpa': 3.5, 'role': 'Project Engineer', 'branches_eligible': 'All branches', 'year': 2024},
    {'company': 'L&T', 'package_lpa': 5.0, 'role': 'Graduate Engineer', 'branches_eligible': 'ME, CE, EE', 'year': 2024},
    {'company': 'Amazon', 'package_lpa': 18.0, 'role': 'SDE', 'branches_eligible': 'CSE, IT', 'year': 2024},
    {'company': 'Capgemini', 'package_lpa': 4.0, 'role': 'Analyst', 'branches_eligible': 'All branches', 'year': 2024},
]
pd.DataFrame(placements).to_csv('placements.csv', index=False)
print("Placements data created.")

# 15. GRIEVANCE CELL
grievance = [{'committee': 'Student Grievance Cell',
    'head': 'Dr. P.K. Verma',
    'contact': '9876500010',
    'email': 'grievance@university.ac.in',
    'timing': '10 AM to 4 PM, Monday to Friday',
    'process': 'Submit written complaint to cell, response within 7 working days',
    'types_handled': 'Academic, Hostel, Harassment, Fee related'}]
pd.DataFrame(grievance).to_csv('grievance.csv', index=False)
print("Grievance cell data created.")

# 16. HEALTH CENTER
health = [{'facility': 'University Health Center',
    'doctor': 'Dr. A.K. Singh',
    'timing': '9 AM to 5 PM weekdays, 9 AM to 1 PM Saturday',
    'emergency': '24x7 nurse available',
    'contact': '9876500020',
    'services': 'First aid, general consultation, ambulance facility',
    'ambulance': 'Available 24x7',
    'nearest_hospital': 'City Hospital, 2 km'}]
pd.DataFrame(health).to_csv('health.csv', index=False)
print("Health center data created.")

# 17. RULES
rules = [
    {'rule': 'Attendance', 'description': 'Minimum 75% attendance required in each subject'},
    {'rule': 'Dress Code', 'description': 'Formal dress code on Monday and Tuesday, ID card mandatory daily'},
    {'rule': 'Mobile Policy', 'description': 'Mobile phones not allowed in classrooms and labs'},
    {'rule': 'Library', 'description': 'Maintain silence, no food or drinks inside'},
    {'rule': 'Ragging', 'description': 'Strictly prohibited, leads to immediate expulsion'},
]
pd.DataFrame(rules).to_csv('rules.csv', index=False)
print("Rules and regulations data created.")

# 18. LABS
labs = [
    {'lab_name': 'Computer Lab 1', 'capacity': 60, 'systems': 60, 'software': 'Windows 10, Python, Java, MS Office', 'location': 'Block B, 1st Floor'},
    {'lab_name': 'Computer Lab 2', 'capacity': 60, 'systems': 60, 'software': 'Linux, C++, MATLAB', 'location': 'Block B, 2nd Floor'},
    {'lab_name': 'Electronics Lab', 'capacity': 30, 'systems': 30, 'software': 'Oscilloscopes, Function Generators', 'location': 'Block C, 1st Floor'},
    {'lab_name': 'Mechanical Workshop', 'capacity': 40, 'systems': 40, 'software': 'Lathe, Drilling machines', 'location': 'Block D, Ground Floor'},
]
pd.DataFrame(labs).to_csv('labs.csv', index=False)
print("Labs and infrastructure data created.")

# 19. IMPORTANT CONTACTS
contacts = [
    {'role': 'Principal', 'name': 'Prof. Vikram Singh', 'contact': '9876500001', 'email': 'principal@university.ac.in'},
    {'role': 'Dean Academics', 'name': 'Dr. S.K. Rao', 'contact': '9876500002', 'email': 'dean@university.ac.in'},
    {'role': 'Dean Student Welfare', 'name': 'Dr. R.K. Tiwari', 'contact': '9876500003', 'email': 'dsw@university.ac.in'},
    {'role': 'Exam Controller', 'name': 'Mr. P.K. Mishra', 'contact': '9876500004', 'email': 'exams@university.ac.in'},
    {'role': 'Security Office', 'name': 'Mr. Balram Singh', 'contact': '9876500005', 'email': 'security@university.ac.in'},
    {'role': 'Emergency', 'name': 'Campus Emergency', 'contact': '112', 'email': 'emergency@university.ac.in'},
]
pd.DataFrame(contacts).to_csv('contacts.csv', index=False)
print("Important contacts data created.")




# 20.  Subjects and Syllabus
import pandas as pd

subjects = []

syllabus = {
    "CSE": {
        1: ["Mathematics I", "Physics", "Basic Electronics", "Programming in C", "English Communication"],
        2: ["Mathematics II", "Chemistry", "Data Structures", "Digital Logic Design", "Environmental Science"],
        3: ["Discrete Mathematics", "Object Oriented Programming", "Computer Organization", "Database Management Systems", "Operating Systems"],
        4: ["Theory of Computation", "Computer Networks", "Software Engineering", "Design and Analysis of Algorithms", "Web Technologies"],
        5: ["Artificial Intelligence", "Machine Learning", "Compiler Design", "Cloud Computing", "Elective I"],
        6: ["Deep Learning", "Information Security", "Mobile Computing", "Big Data Analytics", "Elective II"],
        7: ["Natural Language Processing", "Internet of Things", "Project Management", "Elective III", "Minor Project"],
        8: ["Major Project", "Entrepreneurship", "Professional Ethics", "Elective IV"],
    },
    "ECE": {
        1: ["Mathematics I", "Physics", "Basic Electronics", "Programming in C", "English Communication"],
        2: ["Mathematics II", "Chemistry", "Network Analysis", "Electronic Devices", "Environmental Science"],
        3: ["Signals and Systems", "Analog Electronics", "Digital Electronics", "Electromagnetic Theory", "Mathematics III"],
        4: ["Communication Systems", "Microprocessors", "Control Systems", "VLSI Design", "Linear Integrated Circuits"],
        5: ["Digital Signal Processing", "Wireless Communication", "Embedded Systems", "Antenna Theory", "Elective I"],
        6: ["Optical Communication", "Radar and Television", "Digital Image Processing", "Satellite Communication", "Elective II"],
        7: ["Advanced Communication", "IoT and Applications", "Project Management", "Elective III", "Minor Project"],
        8: ["Major Project", "Entrepreneurship", "Professional Ethics", "Elective IV"],
    },
    "ME": {
        1: ["Mathematics I", "Physics", "Engineering Drawing", "Workshop Practice", "English Communication"],
        2: ["Mathematics II", "Chemistry", "Thermodynamics", "Engineering Mechanics", "Environmental Science"],
        3: ["Fluid Mechanics", "Manufacturing Processes", "Material Science", "Strength of Materials", "Mathematics III"],
        4: ["Heat Transfer", "Machine Design", "Kinematics of Machines", "Metrology", "Industrial Engineering"],
        5: ["Dynamics of Machines", "CAD/CAM", "Refrigeration and AC", "Automobile Engineering", "Elective I"],
        6: ["Finite Element Analysis", "Robotics", "Power Plant Engineering", "Operations Research", "Elective II"],
        7: ["Advanced Manufacturing", "Product Design", "Project Management", "Elective III", "Minor Project"],
        8: ["Major Project", "Entrepreneurship", "Professional Ethics", "Elective IV"],
    },
    "CE": {
        1: ["Mathematics I", "Physics", "Engineering Drawing", "Building Materials", "English Communication"],
        2: ["Mathematics II", "Chemistry", "Surveying", "Mechanics of Solids", "Environmental Science"],
        3: ["Fluid Mechanics", "Structural Analysis", "Concrete Technology", "Soil Mechanics", "Mathematics III"],
        4: ["Design of Structures", "Transportation Engineering", "Hydraulics", "Environmental Engineering", "Geotechnical Engineering"],
        5: ["Foundation Engineering", "Water Resources", "Construction Management", "Remote Sensing", "Elective I"],
        6: ["Bridge Engineering", "Urban Planning", "Earthquake Engineering", "Pavement Design", "Elective II"],
        7: ["Advanced Structures", "Smart Infrastructure", "Project Management", "Elective III", "Minor Project"],
        8: ["Major Project", "Entrepreneurship", "Professional Ethics", "Elective IV"],
    },
    "EE": {
        1: ["Mathematics I", "Physics", "Basic Electronics", "Circuit Theory", "English Communication"],
        2: ["Mathematics II", "Chemistry", "Electrical Machines I", "Network Theory", "Environmental Science"],
        3: ["Electrical Machines II", "Power Systems I", "Control Systems", "Measurements", "Mathematics III"],
        4: ["Power Systems II", "Power Electronics", "Microprocessors", "Switchgear and Protection", "Utilization of Electrical Energy"],
        5: ["High Voltage Engineering", "Electric Drives", "Renewable Energy", "PLC and SCADA", "Elective I"],
        6: ["Smart Grid", "Embedded Systems", "Industrial Automation", "Power Quality", "Elective II"],
        7: ["Advanced Power Systems", "Energy Management", "Project Management", "Elective III", "Minor Project"],
        8: ["Major Project", "Entrepreneurship", "Professional Ethics", "Elective IV"],
    },
    "IT": {
        1: ["Mathematics I", "Physics", "Basic Electronics", "Programming in C", "English Communication"],
        2: ["Mathematics II", "Chemistry", "Data Structures", "Digital Logic", "Environmental Science"],
        3: ["Discrete Mathematics", "Java Programming", "Database Systems", "Computer Networks", "Operating Systems"],
        4: ["Software Engineering", "Web Development", "Theory of Computation", "Design Patterns", "Information Security"],
        5: ["Cloud Computing", "Machine Learning", "Mobile Application Development", "Data Mining", "Elective I"],
        6: ["DevOps", "Blockchain Technology", "Natural Language Processing", "Cyber Security", "Elective II"],
        7: ["Full Stack Development", "AI Applications", "Project Management", "Elective III", "Minor Project"],
        8: ["Major Project", "Entrepreneurship", "Professional Ethics", "Elective IV"],
    },
    "CH": {
        1: ["Mathematics I", "Physics", "Chemistry I", "Engineering Drawing", "English Communication"],
        2: ["Mathematics II", "Chemistry II", "Thermodynamics", "Fluid Mechanics", "Environmental Science"],
        3: ["Chemical Reaction Engineering", "Heat Transfer", "Mass Transfer", "Process Calculations", "Mathematics III"],
        4: ["Process Equipment Design", "Instrumentation", "Chemical Technology", "Transport Phenomena", "Industrial Chemistry"],
        5: ["Petroleum Refining", "Polymer Technology", "Process Control", "Safety Engineering", "Elective I"],
        6: ["Biochemical Engineering", "Environmental Engineering", "Plant Design", "Catalysis", "Elective II"],
        7: ["Advanced Process Design", "Green Chemistry", "Project Management", "Elective III", "Minor Project"],
        8: ["Major Project", "Entrepreneurship", "Professional Ethics", "Elective IV"],
    },
}

for branch, semesters in syllabus.items():
    for sem, subject_list in semesters.items():
        for subject in subject_list:
            subjects.append({
                'branch': branch,
                'semester': sem,
                'subject': subject,
                'summary': f'{subject} is taught in semester {sem} of {branch} branch'
            })

pd.DataFrame(subjects).to_csv('subjects.csv', index=False)
print(f"Subjects and syllabus data created: {len(subjects)} records")

print("\nAll 17 data modules generated successfully")


### Phase 2a — Data Generation (Module 20 — Subjects & Syllabus)

Here I add subjects and syllabus data for all 7 branches across 8 semesters. Each semester has a grouped summary row which helps the RAG system retrieve all subjects for a specific semester accurately when queried.


In [ ]:
### Phase 2a— Data Generation (Module 20 — Subjects & Syllabus)
import pandas as pd

subjects = []
syllabus = {
    "CSE": {
        1: ["Mathematics I", "Physics", "Basic Electronics", "Programming in C", "English Communication"],
        2: ["Mathematics II", "Chemistry", "Data Structures", "Digital Logic Design", "Environmental Science"],
        3: ["Discrete Mathematics", "Object Oriented Programming", "Computer Organization", "Database Management Systems", "Operating Systems"],
        4: ["Theory of Computation", "Computer Networks", "Software Engineering", "Design and Analysis of Algorithms", "Web Technologies"],
        5: ["Artificial Intelligence", "Machine Learning", "Compiler Design", "Cloud Computing", "Elective I"],
        6: ["Deep Learning", "Information Security", "Mobile Computing", "Big Data Analytics", "Elective II"],
        7: ["Natural Language Processing", "Internet of Things", "Project Management", "Elective III", "Minor Project"],
        8: ["Major Project", "Entrepreneurship", "Professional Ethics", "Elective IV"],
    },
    "ECE": {
        1: ["Mathematics I", "Physics", "Basic Electronics", "Programming in C", "English Communication"],
        2: ["Mathematics II", "Chemistry", "Network Analysis", "Electronic Devices", "Environmental Science"],
        3: ["Signals and Systems", "Analog Electronics", "Digital Electronics", "Electromagnetic Theory", "Mathematics III"],
        4: ["Communication Systems", "Microprocessors", "Control Systems", "VLSI Design", "Linear Integrated Circuits"],
        5: ["Digital Signal Processing", "Wireless Communication", "Embedded Systems", "Antenna Theory", "Elective I"],
        6: ["Optical Communication", "Radar and Television", "Digital Image Processing", "Satellite Communication", "Elective II"],
        7: ["Advanced Communication", "IoT and Applications", "Project Management", "Elective III", "Minor Project"],
        8: ["Major Project", "Entrepreneurship", "Professional Ethics", "Elective IV"],
    },
    "ME": {
        1: ["Mathematics I", "Physics", "Engineering Drawing", "Workshop Practice", "English Communication"],
        2: ["Mathematics II", "Chemistry", "Thermodynamics", "Engineering Mechanics", "Environmental Science"],
        3: ["Fluid Mechanics", "Manufacturing Processes", "Material Science", "Strength of Materials", "Mathematics III"],
        4: ["Heat Transfer", "Machine Design", "Kinematics of Machines", "Metrology", "Industrial Engineering"],
        5: ["Dynamics of Machines", "CAD/CAM", "Refrigeration and AC", "Automobile Engineering", "Elective I"],
        6: ["Finite Element Analysis", "Robotics", "Power Plant Engineering", "Operations Research", "Elective II"],
        7: ["Advanced Manufacturing", "Product Design", "Project Management", "Elective III", "Minor Project"],
        8: ["Major Project", "Entrepreneurship", "Professional Ethics", "Elective IV"],
    },
    "CE": {
        1: ["Mathematics I", "Physics", "Engineering Drawing", "Building Materials", "English Communication"],
        2: ["Mathematics II", "Chemistry", "Surveying", "Mechanics of Solids", "Environmental Science"],
        3: ["Fluid Mechanics", "Structural Analysis", "Concrete Technology", "Soil Mechanics", "Mathematics III"],
        4: ["Design of Structures", "Transportation Engineering", "Hydraulics", "Environmental Engineering", "Geotechnical Engineering"],
        5: ["Foundation Engineering", "Water Resources", "Construction Management", "Remote Sensing", "Elective I"],
        6: ["Bridge Engineering", "Urban Planning", "Earthquake Engineering", "Pavement Design", "Elective II"],
        7: ["Advanced Structures", "Smart Infrastructure", "Project Management", "Elective III", "Minor Project"],
        8: ["Major Project", "Entrepreneurship", "Professional Ethics", "Elective IV"],
    },
    "EE": {
        1: ["Mathematics I", "Physics", "Basic Electronics", "Circuit Theory", "English Communication"],
        2: ["Mathematics II", "Chemistry", "Electrical Machines I", "Network Theory", "Environmental Science"],
        3: ["Electrical Machines II", "Power Systems I", "Control Systems", "Measurements", "Mathematics III"],
        4: ["Power Systems II", "Power Electronics", "Microprocessors", "Switchgear and Protection", "Utilization of Electrical Energy"],
        5: ["High Voltage Engineering", "Electric Drives", "Renewable Energy", "PLC and SCADA", "Elective I"],
        6: ["Smart Grid", "Embedded Systems", "Industrial Automation", "Power Quality", "Elective II"],
        7: ["Advanced Power Systems", "Energy Management", "Project Management", "Elective III", "Minor Project"],
        8: ["Major Project", "Entrepreneurship", "Professional Ethics", "Elective IV"],
    },
    "IT": {
        1: ["Mathematics I", "Physics", "Basic Electronics", "Programming in C", "English Communication"],
        2: ["Mathematics II", "Chemistry", "Data Structures", "Digital Logic", "Environmental Science"],
        3: ["Discrete Mathematics", "Java Programming", "Database Systems", "Computer Networks", "Operating Systems"],
        4: ["Software Engineering", "Web Development", "Theory of Computation", "Design Patterns", "Information Security"],
        5: ["Cloud Computing", "Machine Learning", "Mobile Application Development", "Data Mining", "Elective I"],
        6: ["DevOps", "Blockchain Technology", "Natural Language Processing", "Cyber Security", "Elective II"],
        7: ["Full Stack Development", "AI Applications", "Project Management", "Elective III", "Minor Project"],
        8: ["Major Project", "Entrepreneurship", "Professional Ethics", "Elective IV"],
    },
    "CH": {
        1: ["Mathematics I", "Physics", "Chemistry I", "Engineering Drawing", "English Communication"],
        2: ["Mathematics II", "Chemistry II", "Thermodynamics", "Fluid Mechanics", "Environmental Science"],
        3: ["Chemical Reaction Engineering", "Heat Transfer", "Mass Transfer", "Process Calculations", "Mathematics III"],
        4: ["Process Equipment Design", "Instrumentation", "Chemical Technology", "Transport Phenomena", "Industrial Chemistry"],
        5: ["Petroleum Refining", "Polymer Technology", "Process Control", "Safety Engineering", "Elective I"],
        6: ["Biochemical Engineering", "Environmental Engineering", "Plant Design", "Catalysis", "Elective II"],
        7: ["Advanced Process Design", "Green Chemistry", "Project Management", "Elective III", "Minor Project"],
        8: ["Major Project", "Entrepreneurship", "Professional Ethics", "Elective IV"],
    },
}

for branch, semesters in syllabus.items():
    for sem, subject_list in semesters.items():
        # Add one grouped summary row per semester — this is what gets retrieved
        subjects_str = ", ".join(subject_list)
        subjects.append({
            'branch': branch,
            'semester': sem,
            'subject': subjects_str,
            'summary': f'Subjects in semester {sem} of {branch} branch are: {subjects_str}'
        })
        # Also add individual rows for specific subject queries
        for subject in subject_list:
            subjects.append({
                'branch': branch,
                'semester': sem,
                'subject': subject,
                'summary': f'{subject} is a subject in semester {sem} of {branch} branch'
            })

pd.DataFrame(subjects).to_csv('subjects.csv', index=False)
print(f"Subjects and syllabus data created: {len(subjects)} records")

### Phase 3 — Exploratory Data Analysis (EDA)

Before building any model, it is important to understand the data. Here I perform EDA on the student dataset to explore patterns and distributions. This includes looking at how students are distributed across branches, gender ratio, CGPA distribution, attendance patterns and how many students have scholarships.

EDA helps us understand the quality of our data and gives useful insights about the student population.

In [ ]:
# ============================================================
# PHASE 3 — EXPLORATORY DATA ANALYSIS (EDA)
# Dataset: University Query Management System
# ============================================================

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('students.csv')

print("=" * 50)
print("DATASET SHAPE")
print("=" * 50)
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")

print("\n" + "=" * 50)
print("FIRST 5 ROWS")
print("=" * 50)
print(df.head())

print("\n" + "=" * 50)
print("COLUMN NAMES AND DATA TYPES")
print("=" * 50)
print(df.dtypes)

print("\n" + "=" * 50)
print("BASIC STATISTICS")
print("=" * 50)
print(df.describe())

print("\n" + "=" * 50)
print("MISSING VALUES PER COLUMN")
print("=" * 50)
print(df.isnull().sum())

plt.figure(figsize=(8, 4))
df['branch'].value_counts().plot(kind='bar', color='steelblue', edgecolor='black')
plt.title('Number of Students per Branch')
plt.xlabel('Branch')
plt.ylabel('Number of Students')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('branch_distribution.png', dpi=150)
plt.show()
print("Branch distribution chart saved.")

plt.figure(figsize=(5, 5))
df['gender'].value_counts().plot(kind='pie', autopct='%1.1f%%',
    colors=['steelblue', 'salmon'], startangle=90)
plt.title('Gender Distribution')
plt.ylabel('')
plt.tight_layout()
plt.savefig('gender_distribution.png', dpi=150)
plt.show()
print("Gender distribution chart saved.")

plt.figure(figsize=(8, 4))
plt.hist(df['cgpa'], bins=20, color='steelblue', edgecolor='black')
plt.title('CGPA Distribution of Students')
plt.xlabel('CGPA')
plt.ylabel('Number of Students')
plt.tight_layout()
plt.savefig('cgpa_distribution.png', dpi=150)
plt.show()
print("CGPA distribution chart saved.")

plt.figure(figsize=(8, 4))
plt.hist(df['attendance'], bins=20, color='darkorange', edgecolor='black')
plt.title('Attendance Distribution of Students')
plt.xlabel('Attendance (%)')
plt.ylabel('Number of Students')
plt.tight_layout()
plt.savefig('attendance_distribution.png', dpi=150)
plt.show()
print("Attendance distribution chart saved.")

plt.figure(figsize=(8, 4))
df['hostel'].value_counts().plot(kind='bar', color='teal', edgecolor='black')
plt.title('Hostel vs Day Scholar Distribution')
plt.xlabel('Accommodation Type')
plt.ylabel('Number of Students')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('hostel_distribution.png', dpi=150)
plt.show()
print("Hostel distribution chart saved.")

plt.figure(figsize=(5, 5))
df['scholarship'].value_counts().plot(kind='pie', autopct='%1.1f%%',
    colors=['gold', 'lightgray'], startangle=90)
plt.title('Scholarship Distribution')
plt.ylabel('')
plt.tight_layout()
plt.savefig('scholarship_distribution.png', dpi=150)
plt.show()
print("Scholarship distribution chart saved.")

plt.figure(figsize=(8, 4))
df.groupby('branch')['cgpa'].mean().sort_values(ascending=False).plot(
    kind='bar', color='purple', edgecolor='black')
plt.title('Average CGPA per Branch')
plt.xlabel('Branch')
plt.ylabel('Average CGPA')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('avg_cgpa_branch.png', dpi=150)
plt.show()
print("Average CGPA per branch chart saved.")

print("\n" + "=" * 50)
print("EDA COMPLETE - All charts saved successfully")
print("=" * 50)

### Phase 4 — Building the RAG System

This is the core of the project. The RAG pipeline works in the following way:

1. All 16 CSV files are loaded and converted into text documents
2. Each document is converted into a vector using the HuggingFace sentence-transformer model (all-MiniLM-L6-v2)
3. These vectors are stored in a FAISS index for fast retrieval
4. When a user asks a question, it is also converted into a vector
5. FAISS finds the most similar documents to the question
6. These documents are passed to the Groq LLM as context
7. The LLM generates a clear answer and cites the source

This approach ensures that answers are always grounded in actual university data rather than made up by the LLM.

In [ ]:
# ============================================================
# PHASE 4 — RAG SYSTEM BUILD
# Embeddings : sentence-transformers (proper vector embeddings)
# Vector DB  : FAISS
# LLM        : Groq API (free, fast)
# ============================================================

import pandas as pd
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from groq import Groq

# ── STEP 1: SET GROQ API KEY ─────────────────────────────────
client = Groq(api_key="Place_your_Groq_API_Key_here")  # Add your Groq API key here

# ── STEP 2: LOAD SENTENCE TRANSFORMER MODEL ──────────────────
print("Loading sentence-transformer embedding model...")
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
print("Embedding model loaded.")

# ── STEP 3: LOAD ALL CSV FILES ───────────────────────────────
csv_files = [
    'students.csv', 'hostel.csv', 'mess.csv', 'anti_ragging.csv',
    'academic_info.csv', 'exams.csv', 'faculty.csv', 'fees.csv',
    'admissions.csv', 'scholarships.csv', 'library.csv', 'clubs.csv',
    'transport.csv', 'placements.csv', 'grievance.csv', 'health.csv',
    'rules.csv', 'labs.csv', 'contacts.csv', 'subjects.csv'
]

documents = []
sources = []

for file in csv_files:
    try:
        df = pd.read_csv(file)
        source_name = file.replace('.csv', '')
        for _, row in df.iterrows():
            text = ' | '.join([f"{col}: {val}" for col, val in row.items()])
            documents.append(text)
            sources.append(source_name)
        print(f"Loaded: {file} ({len(df)} records)")
    except FileNotFoundError:
        print(f"Skipped: {file}")

print(f"\nTotal documents loaded: {len(documents)}")

# ── STEP 4: CREATE EMBEDDINGS ────────────────────────────────
print("\nCreating embeddings...")
embeddings = embedding_model.encode(documents, show_progress_bar=True)
embeddings = np.array(embeddings).astype('float32')
print(f"Embeddings created: {embeddings.shape}")

# ── STEP 5: STORE IN FAISS ───────────────────────────────────
print("\nStoring in FAISS index...")
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)
print(f"Total documents stored in FAISS: {index.ntotal}")

# ── STEP 6: RAG QUERY FUNCTION ───────────────────────────────

def query_rag(question, n_results=15):
    # Translate Hindi queries to English for better retrieval
    translate_prompt = f"""If the following question is in Hindi, translate it to English.
    If it is already in English, return it as is. Return only the translated question, nothing else.
    Question: {question}"""

    translation = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": translate_prompt}]
    )
    search_question = translation.choices[0].message.content.strip()

    # Use translated question for embedding and retrieval
    question_embedding = embedding_model.encode([search_question])
    question_embedding = np.array(question_embedding).astype('float32')
    distances, indices = index.search(question_embedding, n_results)
    retrieved_docs = [documents[i] for i in indices[0]]
    retrieved_sources = [sources[i] for i in indices[0]]
    unique_sources = list(set(retrieved_sources))
    context = "\n".join(retrieved_docs)

    prompt = f"""You are a helpful university query assistant for an engineering college.
Use ONLY the context provided below to answer the question accurately.
The context contains real university data — trust it completely.

Context:
{context}

Question: {question}

Answering rules:
- If the user specifies a format like bullet points, numbered points, table or single line — follow it exactly
- If the user specifies a number of points like 3 or 4 — give exactly that many
- If the user asks in Hindi — answer in Hindi
- If no format is specified — give a clear factual answer in 2 to 3 sentences
- Never say information is not available if it exists in the context above
- Always give the actual data values from the context — phone numbers, timings, names etc

End with:
Source: {', '.join(unique_sources)}"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

# ── STEP 7: TEST THE RAG SYSTEM ──────────────────────────────
print("\n" + "=" * 50)
print("TESTING RAG SYSTEM")
print("=" * 50)

test_queries = [
    "What is the anti-ragging helpline number?",
    "What is the mess menu for Monday?",
    "Who is the HOD of CSE department?",
    "What is the library timing?",
    "Which companies came for placements?"
]

for query in test_queries:
    print(f"\nQuery: {query}")
    answer = query_rag(query)
    print(f"Answer: {answer}")
    print("-" * 40)

print("\nRAG system working successfully.")

### Phase 5 — Saving Files to Google Drive

Since Google Colab resets the runtime after some time, all generated CSV files are saved to Google Drive. This way the data is not lost and can be reloaded quickly without regenerating everything from scratch.

In [ ]:

# ============================================================
# PHASE 5 — SAVING TO GOOGLE DRIVE
# Saving all CSV files and embeddings for persistence
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import os
import shutil
import numpy as np

# Create folder in Drive
os.makedirs('/content/drive/MyDrive/UniversityRAG', exist_ok=True)

# Save all CSV files
csv_files = [
    'students.csv', 'hostel.csv', 'mess.csv', 'anti_ragging.csv',
    'academic_info.csv', 'exams.csv', 'faculty.csv', 'fees.csv',
    'admissions.csv', 'scholarships.csv', 'library.csv', 'clubs.csv',
    'transport.csv', 'placements.csv', 'grievance.csv', 'health.csv',
    'rules.csv', 'labs.csv', 'contacts.csv'
]

for file in csv_files:
    shutil.copy(file, f'/content/drive/MyDrive/UniversityRAG/{file}')

# Save embeddings
np.save('/content/drive/MyDrive/UniversityRAG/embeddings.npy', embeddings)
import faiss
faiss.write_index(index, '/content/drive/MyDrive/UniversityRAG/faiss_index.bin')

print("All files saved to Google Drive successfully.")

### Phase 6 — Streamlit App and Deployment

The final step is creating a simple web interface using Streamlit so that anyone can use the query system without running Python code. The app takes a question as input, runs it through the RAG pipeline and displays the answer along with the source of the information.

The app is deployed on Hugging Face Spaces and is publicly accessible.

**Deployment link:** https://huggingface.co/spaces/adeemk/university-query-management-system

**GitHub link:** https://github.com/adeemkhan42004-byte/university-query-management-system-

In [ ]:
# ============================================================
# PHASE 6 — STREAMLIT UI
# Chatbot interface with chat history, word animation,
# relevance score, Hindi support and background themes
# ============================================================

streamlit_code = '''import streamlit as st
import pandas as pd
import numpy as np
import faiss
import os
import time
from sentence_transformers import SentenceTransformer
from groq import Groq

st.set_page_config(
    page_title="University Query Management System",
    page_icon="🎓",
    layout="wide"
)

if "bg_color" not in st.session_state:
    st.session_state.bg_color = "#0E1117"

with st.sidebar:
    st.header("Settings")
    bg_choice = st.selectbox(
        "Background Theme",
        ["Dark", "Navy", "Charcoal", "Deep Purple", "Forest"]
    )
    bg_map = {
        "Dark": "#0E1117",
        "Navy": "#0A2342",
        "Charcoal": "#1E1E1E",
        "Deep Purple": "#1A0E2E",
        "Forest": "#0E2A1E"
    }
    st.session_state.bg_color = bg_map[bg_choice]
    st.markdown("---")
    st.caption("University Query Management System")
    st.caption("Built with RAG + Groq LLM")
    if st.button("Clear Chat History"):
        st.session_state.messages = []
        st.rerun()

st.markdown(f"""
<style>
    .stApp {{
        background-color: {st.session_state.bg_color};
    }}
    .main-title {{
        text-align: center;
        color: #00C9A7;
        font-size: 32px;
        font-weight: bold;
        margin-bottom: 0px;
    }}
    .sub-title {{
        text-align: center;
        color: #94A3B8;
        font-size: 14px;
        margin-top: 0px;
    }}
</style>
""", unsafe_allow_html=True)

st.markdown(\'<p class="main-title">University Query Management System</p>\', unsafe_allow_html=True)
st.markdown(\'<p class="sub-title">Ask anything about hostel, mess, fees, placements, subjects, anti-ragging and more</p>\', unsafe_allow_html=True)
st.markdown("---")

@st.cache_resource
def load_rag():
    api_key = os.environ.get("Place_your_Groq_API_Key_here", "")

    client = Groq(api_key=api_key)
    embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

    csv_files = [
        "students.csv", "hostel.csv", "mess.csv", "anti_ragging.csv",
        "academic_info.csv", "exams.csv", "faculty.csv", "fees.csv",
        "admissions.csv", "scholarships.csv", "library.csv", "clubs.csv",
        "transport.csv", "placements.csv", "grievance.csv", "health.csv",
        "rules.csv", "labs.csv", "contacts.csv", "subjects.csv"
    ]

    documents = []
    sources = []
    for file in csv_files:
        try:
            df = pd.read_csv(file)
            source_name = file.replace(".csv", "")
            for _, row in df.iterrows():
                text = " | ".join([f"{col}: {val}" for col, val in row.items()])
                documents.append(text)
                sources.append(source_name)
        except FileNotFoundError:
            pass

    embeddings = embedding_model.encode(documents)
    embeddings = np.array(embeddings).astype("float32")
    dimension = embeddings.shape[1]
    index = faiss.IndexFlatL2(dimension)
    index.add(embeddings)
    return client, embedding_model, index, documents, sources

client, embedding_model, index, documents, sources = load_rag()

if "messages" not in st.session_state:
    st.session_state.messages = []

def query_rag(question, chat_history):
    translate_prompt = f"""If the following question is in Hindi, translate it to English.
If it is already in English, return it as is. Return only the question, nothing else.
Question: {question}"""
    translation = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": translate_prompt}]
    )
    search_question = translation.choices[0].message.content.strip()

    question_embedding = embedding_model.encode([search_question])
    question_embedding = np.array(question_embedding).astype("float32")
    distances, indices = index.search(question_embedding, 15)

    retrieved_docs = [documents[i] for i in indices[0]]
    retrieved_sources = [sources[i] for i in indices[0]]
    unique_sources = list(set(retrieved_sources))
    context = "\\n".join(retrieved_docs)

    best_distance = float(distances[0][0])
    relevance = max(0, min(100, int(100 - (best_distance * 8))))

    history_text = ""
    recent = chat_history[-12:]
    for msg in recent:
        role = "User" if msg["role"] == "user" else "Assistant"
        history_text += f"{role}: {msg[\'content\']}\\n"

    prompt = f"""You are a helpful university query assistant for an engineering college.
Use ONLY the context provided below to answer the question accurately.
The context contains real university data — trust it completely.

Previous conversation (use this to understand follow-up questions):
{history_text}

Context:
{context}

Current Question: {question}

Answering rules:
- Use the previous conversation to understand follow-up questions and pronouns like it, that, same
- If the user specifies a format like bullet points, numbered points, table or single line — follow it exactly
- If the user specifies a number of points like 3 or 4 — give exactly that many
- If the user asks in Hindi — answer in Hindi
- If no format is specified — give a clear factual answer in 2 to 3 sentences
- Never say information is not available if it exists in the context above
- Always give the actual data values — phone numbers, timings, names etc

End with:
Source: {", ".join(unique_sources)}"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content, relevance

for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.markdown(msg["content"])
        if msg["role"] == "assistant" and "relevance" in msg:
            st.caption(f"Relevance Score: {msg[\'relevance\']}%")

if prompt_text := st.chat_input("Type your question here... (or use your keyboard mic button for voice input)"):
    st.session_state.messages.append({"role": "user", "content": prompt_text})
    with st.chat_message("user"):
        st.markdown(prompt_text)

    with st.chat_message("assistant"):
        with st.spinner("Finding answer..."):
            answer, relevance = query_rag(prompt_text, st.session_state.messages)

        placeholder = st.empty()
        displayed = ""
        for word in answer.split(" "):
            displayed += word + " "
            placeholder.markdown(displayed)
            time.sleep(0.03)

        st.caption(f"Relevance Score: {relevance}%")

    st.session_state.messages.append({
        "role": "assistant",
        "content": answer,
        "relevance": relevance
    })
'''

with open("app.py", "w") as f:
    f.write(streamlit_code)

print("app.py created successfully.")

### Phase 7 — Deployment Files

Here I create the necessary files for deploying the application on Hugging Face Spaces. This includes the requirements.txt file which lists all the Python libraries needed to run the app, and the app.py Streamlit file which is the actual web interface.

The app.py file uses an environment variable for the API key instead of hardcoding it directly in the code. This is a security best practice so that the key is never exposed publicly on GitHub or Hugging Face.

In [ ]:
# ============================================================
# PHASE 7 — DEPLOYMENT FILES
# Creating requirements.txt and packages.txt for Hugging Face
# ============================================================

# requirements.txt — Python libraries
requirements = """streamlit
faiss-cpu
sentence-transformers
groq
pandas
numpy
"""

with open("requirements.txt", "w") as f:
    f.write(requirements)
print("requirements.txt created.")

# Copy all CSV files list for reference
import os
files = os.listdir('.')
print("\nFiles ready for deployment:")
for f in sorted(files):
    if f.endswith('.csv') or f.endswith('.py') or f.endswith('.txt'):
        print(f" - {f}")

### Phase 8 — Downloading Project Files

Here I package all the project files including the Streamlit app, requirements file and all 16 CSV data modules into a single zip file and download it to my local machine. These files are then uploaded to Hugging Face Spaces and GitHub for deployment and submission.

In [ ]:
# ============================================================
# PHASE 8 — DOWNLOADING PROJECT FILES
# Packaging all files into a zip for upload to
# Hugging Face Spaces and GitHub
# ============================================================

import zipfile
import os
from google.colab import files

# Create zip with all required files
with zipfile.ZipFile('university_rag_deploy.zip', 'w') as zipf:
    # Add app.py
    zipf.write('app.py')
    # Add requirements.txt
    zipf.write('requirements.txt')
    # Add all CSV files
    csv_files = [
        'students.csv', 'hostel.csv', 'mess.csv', 'anti_ragging.csv',
        'academic_info.csv', 'exams.csv', 'faculty.csv', 'fees.csv',
        'admissions.csv', 'scholarships.csv', 'library.csv', 'clubs.csv',
        'transport.csv', 'placements.csv', 'grievance.csv', 'health.csv',
        'rules.csv', 'labs.csv', 'contacts.csv'
    ]
    for f in csv_files:
        if os.path.exists(f):
            zipf.write(f)
            print(f"Added: {f}")

print("\nZip created. Downloading...")
files.download('university_rag_deploy.zip')

### Phase 9 — Updating app.py with Environment Variable

For security reasons, the Groq API key should not be hardcoded directly in the code. Here I update the app.py file to read the API key from an environment variable instead. This way the key is stored securely as a secret on Hugging Face Spaces and never exposed publicly in the code on GitHub.

In [ ]:
# ============================================================
# PHASE 9 — UPDATING APP.PY WITH ENVIRONMENT VARIABLE
# Removing hardcoded API key and replacing with
# os.environ.get() for security best practices
# ============================================================
streamlit_code = '''import streamlit as st
import pandas as pd
import numpy as np
import faiss
import os
from sentence_transformers import SentenceTransformer
from groq import Groq

st.set_page_config(
    page_title="University Query Management System",
    page_icon="🎓",
    layout="centered"
)

st.title("University Query Management System")
st.markdown("Ask anything about admissions, hostel, mess, fees, placements, anti-ragging and more.")
st.markdown("---")

@st.cache_resource
def load_rag():
    # API key loaded from environment variable for security
    api_key = os.environ.get("GROQ_API_KEY", "")
    client = Groq(api_key=api_key)
    embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

    csv_files = [
        "students.csv", "hostel.csv", "mess.csv", "anti_ragging.csv",
        "academic_info.csv", "exams.csv", "faculty.csv", "fees.csv",
        "admissions.csv", "scholarships.csv", "library.csv", "clubs.csv",
        "transport.csv", "placements.csv", "grievance.csv", "health.csv",
        "rules.csv", "labs.csv", "contacts.csv"
    ]

    documents = []
    sources = []

    for file in csv_files:
        try:
            df = pd.read_csv(file)
            source_name = file.replace(".csv", "")
            for _, row in df.iterrows():
                text = " | ".join([f"{col}: {val}" for col, val in row.items()])
                documents.append(text)
                sources.append(source_name)
        except FileNotFoundError:
            pass

    embeddings = embedding_model.encode(documents)
    embeddings = np.array(embeddings).astype("float32")
    dimension = embeddings.shape[1]
    index = faiss.IndexFlatL2(dimension)
    index.add(embeddings)

    return client, embedding_model, index, documents, sources

client, embedding_model, index, documents, sources = load_rag()

def query_rag(question, n_results=5):
    question_embedding = embedding_model.encode([question])
    question_embedding = np.array(question_embedding).astype("float32")
    distances, indices = index.search(question_embedding, n_results)
    retrieved_docs = [documents[i] for i in indices[0]]
    retrieved_sources = [sources[i] for i in indices[0]]
    unique_sources = list(set(retrieved_sources))
    context = "\\n".join(retrieved_docs)

    prompt = f"""You are a university query assistant.
Answer ONLY based on the context below. Be specific and direct.
If the exact answer is in the context, state it clearly.
Do not say information is not available if it is present in the context.

Context:
{context}

Question: {question}

Give a direct specific answer in 2-3 sentences maximum.
End with:
Source: {", ".join(unique_sources)}"""

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

st.subheader("Ask a Question")
question = st.text_input("Type your question here:", placeholder="e.g. What is the mess menu for Monday?")

if st.button("Get Answer"):
    if question.strip() == "":
        st.warning("Please enter a question.")
    else:
        with st.spinner("Finding answer..."):
            answer = query_rag(question)
        st.markdown("### Answer")
        st.markdown(answer)

st.markdown("---")
st.markdown("**Sample Questions:**")
st.markdown("- What is the anti-ragging helpline number?")
st.markdown("- What is the mess menu for Monday?")
st.markdown("- Who is the HOD of CSE department?")
st.markdown("- What is the library timing?")
st.markdown("- Which companies came for placements?")
st.markdown("- What is the hostel curfew time?")
st.markdown("- What scholarships are available?")
st.markdown("- What is the fee structure for CSE?")
st.markdown("- How do I report a grievance?")
st.markdown("- What are the transport routes available?")
'''

with open("app.py", "w") as f:
    f.write(streamlit_code)

print("app.py updated with environment variable.")